# Dengue en Loreto (2000 - 2004)

## Notebook 01 - Exploración y calidad de datos

**Pregunta de análisis:** ¿Cuándo, dónde y a quiénes afecta más el dengue en Loreto, y qué cambió en las epidemias de 2023 y 2024?

**Fuente:** Vigilancia epidemiológica de dengue, Centro Nacional de Epidemiología, Prevencion y Control de Enfermedades (CDC Perú - MINSA), Plataforma Nacional de Datos Abiertos.

Este notebook carga el dataset nacional, lo compara con su diccionario de datos, diagnostica y corrige problemas de codificación de texto, valida los datos contra una cifra oficial y filtra los cados de Loreto.

**Contenido**
1. Lectura del archivo
2. Carga de datos
3. Columnas reales frente al diccionario
4. Exploración del dataset nacional
5. Codificación de texto: diagnóstico y corrección
6. Validación
7. Filtro de Loreto
8. Inventario de datos y limitaciones

**Resultado:** `data/processed/dengue_loreto.csv`

> ⚠️ Los datos crudos (103 MB) no se suben a GitHub porque superan el límite de 100 MB. Se descargan desde la Plataforma Nacional de Datos Abiertos y se guardan en `data/raw/`.

## 1. Lectura del archivo

Antes de cargar un archivo grande con pandas, se revisan sus primeras líneas en crudo para identificar su separador y posibles problemas de codificación.

In [1]:
import re
import pandas as pd

ruta = "../data/raw/datos_abiertos_vigilancia_dengue_2000_2024.csv"

with open(ruta, encoding="utf-8", errors="replace") as archivo:
    for _ in range(5):
        print(archivo.readline())

departamento;provincia;distrito;localidad;enfermedad;ano;semana;diagnostic;diresa;ubigeo;localcod;edad;tipo_edad;sexo

LORETO;ALTO AMAZONAS;YURIMAGUAS;000101;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160201;;58;A;M

LORETO;ALTO AMAZONAS;YURIMAGUAS;000101;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160201;;44;A;F

LORETO;ALTO AMAZONAS;YURIMAGUAS;BARRIO MORALILLOS;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160201;;21;A;F

LORETO;MAYNAS;IQUITOS;005902;DENGUE SIN SIGNOS DE ALARMA;2000;1;A97.0;16;160101;;74;A;M



### 1.1 Detección automática de la codificación

Usamos `charset-normalizer`, analiza los bytes del archivoy estima su codificación. Se requiere instalarse en con el entorno activado: `pip install charset-normalizer`.

In [2]:
from charset_normalizer import from_path

deteccion = from_path(ruta).best()
print("Codificación detectada:", deteccion.encoding)

Codificación detectada: utf_8


**Hallazgos de la lectura**
- El separador es punto y coma (`;`), no coma.
- Las columnas reales incluyen `localidad` y `localcod`, que no figuran en el diccionario [Vigilancia Epidemiológica de Dengue](https://www.datosabiertos.gob.pe/dataset/vigilancia-epidemiol%C3%B3gica-de-dengue)

---
## 2. Carga de datos

Según el diccionario, `ubigeo`, `edad` y `diresa` son texto. Se indica explícitamente con `dtype` para que pandas no las convierta a número: los códigos perderían los ceros a la izquierda y la edad se mezclaría con su unidad (`tipo_edad`).

In [3]:
CODIFICACION = "utf-8" 

df = pd.read_csv(
    ruta,
    sep=";",
    encoding=CODIFICACION,
    dtype={"ubigeo": str, "edad": str, "diresa": str, "localcod": str}
)
df.shape

(1029421, 14)

In [4]:
# Usamos "deep" para medir tambien el texto que ocupa mucha memoria en pandas
df.info(memory_usage="deep")


<class 'pandas.DataFrame'>
RangeIndex: 1029421 entries, 0 to 1029420
Data columns (total 14 columns):
 #   Column        Non-Null Count    Dtype
---  ------        --------------    -----
 0   departamento  1029421 non-null  str  
 1   provincia     1029421 non-null  str  
 2   distrito      1029421 non-null  str  
 3   localidad     906823 non-null   str  
 4   enfermedad    1029421 non-null  str  
 5   ano           1029421 non-null  int64
 6   semana        1029421 non-null  int64
 7   diagnostic    1029421 non-null  str  
 8   diresa        1029398 non-null  str  
 9   ubigeo        1029421 non-null  str  
 10  localcod      880780 non-null   str  
 11  edad          1029421 non-null  str  
 12  tipo_edad     1029421 non-null  str  
 13  sexo          1029421 non-null  str  
dtypes: int64(2), str(12)
memory usage: 672.6 MB


In [5]:
df.head()

,departamento,provincia,distrito,localidad,enfermedad,ano,semana,diagnostic,diresa,ubigeo,localcod,edad,tipo_edad,sexo
0,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,58,A,M
1,LORETO,ALTO AMAZONAS,YURIMAGUAS,000101,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,44,A,F
2,LORETO,ALTO AMAZONAS,YURIMAGUAS,BARRIO MORALILLOS,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160201,NaN,21,A,F
3,LORETO,MAYNAS,IQUITOS,005902,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,NaN,74,A,M
4,LORETO,MAYNAS,IQUITOS,CARDOZO,DENGUE SIN SIGNOS DE ALARMA,2000,1,A97.0,16,160101,NaN,42,A,F


**Hallazgos**
- **1,029,421 filas**: cada fila es un caso individual de dengue (hay edad y sexo por fila).
- El archivo pesa 103 MB, pero en memoria ocupa unos 672.6 MB: el texto ocupa mucho más espacio en pandas que en el CSV. Por eso conviene filtrar Loreto pronto.

---
## 3. Columnas reales frente al diccionario

El diccionario de datos del portal describe 13 columnas. Se compara con las columnas reales del archivo.

In [6]:
columnas_diccionario = [
    "departamento", "provincia", "distrito", "enfermedad", "ano", "semana",
    "diagnostic", "tipo_dx", "diresa", "ubigeo", "edad", "tipo_edad", "sexo"
]

print("En el diccionario pero no en el archivo:", set(columnas_diccionario) - set(df.columns))
print("En el archivo pero no en el diccionario:", set(df.columns) - set(columnas_diccionario))

En el diccionario pero no en el archivo: {'tipo_dx'}
En el archivo pero no en el diccionario: {'localcod', 'localidad'}


**Hallazgo:** el diccionario y los datos no coinciden en las dos direcciones.
- **Falta `tipo_dx`**: no es posible distinguir casos confirmados, probables y sospechosos.
- **Sobran `localidad` y `localcod`**, que no están documentadas.

Cuando el diccionario y los datos no coinciden, mandan los datos.

---

## 4. Exploración del dataset nacional

In [7]:
pd.DataFrame({
    "enfermedad": df["enfermedad"].value_counts().values,
    "diagnostic": df["diagnostic"].value_counts().values
}, index=df["enfermedad"].value_counts().index)

,enfermedad,diagnostic
enfermedad,,
DENGUE SIN SIGNOS DE ALARMA,915243,915243
DENGUE CON SIGNOS DE ALARMA,110166,110166
DENGUE GRAVE,4012,4012


In [8]:
(df["enfermedad"].value_counts(normalize=True) * 100).round(1)

enfermedad
DENGUE SIN SIGNOS DE ALARMA    88.9
DENGUE CON SIGNOS DE ALARMA    10.7
DENGUE GRAVE                    0.4
Name: proportion, dtype: float64

In [9]:
# Unidad de la edad: A = años, M = meses, D = días
df["tipo_edad"].value_counts()

tipo_edad
A    1022720
M       6377
D        324
Name: count, dtype: int64

In [10]:
df["ano"].value_counts().sort_index()

ano
2000      5557
2001     23526
2002      8086
2003      3349
2004      9547
2005      5640
2006      4022
2007      6344
2008     12824
2009     13407
2010     16842
2011     28084
2012     28505
2013     13091
2014     17234
2015     35816
2016     25160
2017     68290
2018      4698
2019     15287
2020     47932
2021     44791
2022     63217
2023    256641
2024    271531
Name: count, dtype: int64

In [11]:
# Rango de semanas epidemiologicas: del 1 al 52 o 53
df["semana"].agg(["min", "max"])

min     1
max    53
Name: semana, dtype: int64

In [12]:
df["departamento"].value_counts()

departamento
PIURA            237961
LIMA             122854
LORETO           107074
LA LIBERTAD      100810
ICA               68620
LAMBAYEQUE        56383
UCAYALI           52621
TUMBES            45471
SAN MARTIN        44518
ANCASH            37661
MADRE DE DIOS     32920
CAJAMARCA         29711
JUNIN             26110
AMAZONAS          16805
CUSCO             15210
HUANUCO           12119
AYACUCHO           9946
CALLAO             8138
PASCO              3802
PUNO                647
AREQUIPA             28
APURIMAC             11
MOQUEGUA              1
Name: count, dtype: int64

**Hallazgos**
- `enfermedad` y `diagnostic` tienen exactamente los mismos conteos: una es el nombre y la otra el código CIE-10 (`A97.0`, `A97.1`, `A97.2`). Son redundantes; se usará solo `enfermedad`.
- Forma clínica: **88.9%** sin signos de alarma, **10.7%** con signos de alarma y **0.4%** graves.
- **6,701 casos** (0.65%) tienen la edad en meses o días. Se convertirán a años en la limpieza.
- **2024 (271,531) superó a 2023 (256,641)**. Juntos suman más de la mitad de todos los casos registrados entre 2000 y 2024.
- **Loreto es el tercer departamento** con más casos: 107,074 (10.4% del total nacional), detrás de Piura y Lima.

### 4.1 Filas duplicadas

In [13]:
print(df.duplicated().sum(), "filas identicas")

152737 filas identicas


**Interpretación:** el dataset no tiene un identificador de paciente. Dos personas de la misma edad y sexo, contagiadas en el mismo lugar y la misma semana, producen filas idénticas sin ser un error. Por eso **las filas duplicadas no se eliminan**: no hay forma de distinguir un registro repetido de dos casos reales.

---
## 5. Codificación de texto: diagnóstico y corrección

Cuando un texto se guarda en una codificación y se lee en otra, letras como la ñ o las tildes se dañan. Hay dos tipos de daño, con soluciones distintas:

| Cómo se ve | Causa | Solución |
|---|---|---|
| `MARAÃ‘ON` | Doble codificación: texto UTF-8 leído como cp1252 o latin-1 | Reparación automática |
| `MARA�ON` | El carácter original se perdió | Corrección manual con un diccionario |

### 5.1 Diagnóstico

In [14]:
COLUMNAS_TEXTO = ["departamento", "provincia", "distrito", "localidad", "enfermedad"]

# Marcas típicas de texto dañado: Ã y Â (doble codificación) y � (carácter perdido)
PATRON_DANADO = r"[ÃÂ\ufffd]"

def contar_danados(datos, columnas):
    """Cuenta, por columna, cuántos valores distintos contienen marcas de texto dañado."""
    return {
        col: datos[col].dropna().drop_duplicates().str.contains(PATRON_DANADO, regex=True).sum()
        for col in columnas
    }

contar_danados(df, COLUMNAS_TEXTO)

{'departamento': np.int64(0),
 'provincia': np.int64(4),
 'distrito': np.int64(12),
 'localidad': np.int64(104),
 'enfermedad': np.int64(0)}

In [15]:
# Ver los valores dañados de cada columna
for col in COLUMNAS_TEXTO:
    unicos = pd.Series(df[col].dropna().unique())
    danados = unicos[unicos.str.contains(PATRON_DANADO, regex=True)]
    if len(danados) > 0:
        print(f"{col} ({len(danados)}):", list(danados[:20]))

provincia (4): ['FERRE�AFE', 'DATEM DEL MARA�ON', 'MARA�ON', 'CA�ETE']
distrito (12): ['CU�UMBUQUI', 'PARI�AS', 'FERRE�AFE', 'NEPE�A', 'SA�A', 'I�APARI', 'LA TINGUI�A', 'KOS�IPATA', 'BRE�A', 'CA�ARIS', 'SAN VICENTE DE CA�ETE', 'ZU�IGA']
localidad (104): ['CU�UMBUQUE', '�000101', 'GRAU N� 1692', 'LA CABA�A', 'JR. LIBERTAD N� 212', 'JR. MI PERU N� 188', 'AMAZONAS N� 1092', 'LA PE�A', 'U�A DE GATO', 'A.H. CIUDAD DEL NI�O', 'PARI�AS', 'A.H. �ACARA', 'SECTOR N� 03', 'SECTOR N� 01', 'JR. LOS MANGOS N� 101', 'A.H. �ACARA C 2', 'CA�AVERAL', 'JR. 28 DE JULIO N� 382', 'CRUZ DE CA�A', 'VI�A GRANDE']


In [16]:
# Nombres con caracteres no ASCII (ñ, tildes): para revisar que se vean bien
for col in ["provincia", "distrito"]:
    unicos = pd.Series(df[col].dropna().unique())
    print(col, ":", list(unicos[unicos.str.contains(r"[^\x00-\x7F]", regex=True)]))

provincia : ['FERRE�AFE', 'DATEM DEL MARA�ON', 'MARA�ON', 'CA�ETE']
distrito : ['CU�UMBUQUI', 'PARI�AS', 'FERRE�AFE', 'NEPE�A', 'SA�A', 'I�APARI', 'LA TINGUI�A', 'KOS�IPATA', 'BRE�A', 'CA�ARIS', 'SAN VICENTE DE CA�ETE', 'ZU�IGA']


El dataset presentan varios valores dañados, en las 5 siguientes columnas: **departamento**, **provincia**, **distrito**, **localidad** y **enfermedad**.